# Generative KI: Ein eigenes, winziges Sprachmodell bauen

In diesem Notebook baust du ein stark vereinfachtes Sprachmodell: ein sogenanntes **n-Gramm-Modell**. Es funktioniert nach demselben Grundprinzip wie ein echtes LLM (z. B. hinter ChatGPT oder Claude): Es sagt vorher, welches Wort als Nächstes am wahrscheinlichsten kommt – nur eben nicht mit einem riesigen neuronalen Netz, sondern mit einfachem Abzählen von Wortpaaren.

Wir brauchen dafür keine externe Bibliothek außer der Python-Standardbibliothek (`re`, `random`, `collections`) – kein Internet, kein API-Key, alles läuft offline.

### Schritt 1: Einen Beispieltext festlegen

Wir definieren einen kurzen deutschen Text direkt als Python-String. Achte darauf, dass sich manche Wortfolgen wiederholen – nur dadurch hat unser Modell später überhaupt genug Beispiele, um ein "nächstes Wort" vorherzusagen.

In [ ]:
text = """
Die künstliche Intelligenz verändert die Schule und die Gesellschaft.
Viele Schüler nutzen die künstliche Intelligenz jeden Tag für die Hausaufgaben.
Die künstliche Intelligenz kann Texte schreiben, Bilder erzeugen und Fragen beantworten.
Trotzdem macht die künstliche Intelligenz manchmal Fehler und erfindet Fakten, die es nicht gibt.
Die Schule muss lernen, mit der künstlichen Intelligenz verantwortungsvoll umzugehen.
Viele Lehrer diskutieren über die künstliche Intelligenz im Unterricht.
Die Gesellschaft verändert sich durch die künstliche Intelligenz sehr schnell.
Manche Menschen finden die künstliche Intelligenz spannend, andere finden die künstliche Intelligenz beängstigend.
Am Ende entscheidet der Mensch, wie die künstliche Intelligenz eingesetzt wird.
"""

print(text)

### Schritt 2: Text vorverarbeiten (Tokenisieren)

Bevor wir Wörter zählen können, müssen wir den Text in einzelne Wörter zerlegen ("tokenisieren") und alles klein schreiben, damit z. B. "Die" und "die" als dasselbe Wort gezählt werden. Satzzeichen entfernen wir dabei mit einem regulären Ausdruck (`re`).

In [ ]:
import re
import random
from collections import defaultdict

random.seed(42)

def tokenisiere(text):
    text = text.lower()
    # Nur Buchstaben und Leerzeichen behalten, Satzzeichen entfernen
    text = re.sub(r'[^a-zäöüß\s]', ' ', text)
    woerter = text.split()
    return woerter

woerter = tokenisiere(text)
print(f"Anzahl Wörter im Text: {len(woerter)}")
print("Erste 15 Wörter:", woerter[:15])

### Schritt 3: Bigramme zählen

Ein **Bigramm** ist ein Paar aus einem Wort und dem darauffolgenden Wort. Wir zählen für jedes Wort, welche Wörter im Text jeweils direkt danach vorkommen, und wie oft. Das Ergebnis speichern wir in einem Dictionary: `wort -> Liste der Folgewörter` (ein Folgewort kann mehrfach in der Liste stehen, wenn es mehrfach vorkommt – das bildet die Häufigkeit ab).

In [ ]:
def zaehle_bigramme(woerter):
    folgewoerter = defaultdict(list)
    for i in range(len(woerter) - 1):
        aktuelles_wort = woerter[i]
        naechstes_wort = woerter[i + 1]
        folgewoerter[aktuelles_wort].append(naechstes_wort)
    return folgewoerter

bigramme = zaehle_bigramme(woerter)
print("Anzahl unterschiedlicher Wörter, für die wir Folgewörter kennen:", len(bigramme))

### Aufgabe 1:
Der Code unten zeigt dir die möglichen Folgewörter für das Wort `"die"` an. Das IST bereits die "Wahrscheinlichkeitsverteilung über das nächste Wort", nur eben als einfache Liste statt als Prozentzahlen: Wörter, die öfter vorkommen, tauchen auch öfter in der Liste auf. Probiere danach mindestens zwei weitere Wörter aus dem Text aus (z. B. `"künstliche"` oder `"intelligenz"`), indem du die markierte Zeile änderst.

In [ ]:
test_wort = "die"  # <-- HIER ÄNDERST DU: probiere z. B. "künstliche" oder "intelligenz"

print(f"Mögliche Folgewörter für '{test_wort}':")
print(bigramme[test_wort])

# Häufigkeit der einzelnen Folgewörter zählen
from collections import Counter
haeufigkeiten = Counter(bigramme[test_wort])
print(f"\nHäufigkeiten: {dict(haeufigkeiten)}")

### Schritt 4: Text generieren

Jetzt nutzen wir die Bigramme, um neuen Text zu erzeugen: Wir starten mit einem Wort und wählen zufällig (gewichtet nach Häufigkeit, mit `random.choice` auf der Liste der Folgewörter) das nächste Wort aus. Das wiederholen wir viele Male hintereinander – genau wie ein echtes Sprachmodell Wort für Wort (bzw. Token für Token) generiert.

In [ ]:
def generiere_text(bigramme, startwort, anzahl_woerter=20):
    aktuelles_wort = startwort
    erzeugte_woerter = [aktuelles_wort]

    for _ in range(anzahl_woerter - 1):
        moegliche_folgewoerter = bigramme.get(aktuelles_wort)
        if not moegliche_folgewoerter:
            break  # Kein bekanntes Folgewort mehr - wir hören auf
        aktuelles_wort = random.choice(moegliche_folgewoerter)
        erzeugte_woerter.append(aktuelles_wort)

    return " ".join(erzeugte_woerter)


print(generiere_text(bigramme, startwort="die", anzahl_woerter=20))

Führe die Zelle oben mehrmals aus (Shift+Enter) – da die Auswahl zufällig ist, bekommst du jedes Mal einen etwas anderen Text. Wahrscheinlich klingt er stellenweise sinnvoll, stellenweise aber auch repetitiv oder unsinnig. Das liegt daran, dass unser Modell sich nur das jeweils letzte Wort "merkt" – ein echtes LLM berücksichtigt sehr viel mehr Kontext.

### Aufgabe 2: Von Bigrammen zu Trigrammen

Ein **Trigramm** betrachtet nicht nur ein Wort, sondern zwei aufeinanderfolgende Wörter, um das dritte vorherzusagen ("Wortpaar → nächstes Wort" statt "Wort → nächstes Wort"). Das Modell hat dadurch mehr Kontext und erzeugt meist sinnvolleren Text. Der Code unten ist bereits vollständig – lies ihn genau durch (er ist sehr ähnlich zur Bigramm-Funktion oben, nur dass der Schlüssel jetzt ein *Tupel* aus zwei Wörtern ist) und führe ihn aus.

In [ ]:
def zaehle_trigramme(woerter):
    folgewoerter = defaultdict(list)
    for i in range(len(woerter) - 2):
        wortpaar = (woerter[i], woerter[i + 1])  # Schlüssel: zwei Wörter statt einem
        naechstes_wort = woerter[i + 2]
        folgewoerter[wortpaar].append(naechstes_wort)
    return folgewoerter


def generiere_text_trigramm(trigramme, startwortpaar, anzahl_woerter=20):
    erzeugte_woerter = list(startwortpaar)
    aktuelles_wortpaar = startwortpaar

    for _ in range(anzahl_woerter - 2):
        moegliche_folgewoerter = trigramme.get(aktuelles_wortpaar)
        if not moegliche_folgewoerter:
            break
        naechstes_wort = random.choice(moegliche_folgewoerter)
        erzeugte_woerter.append(naechstes_wort)
        aktuelles_wortpaar = (aktuelles_wortpaar[1], naechstes_wort)

    return " ".join(erzeugte_woerter)


trigramme = zaehle_trigramme(woerter)
print(generiere_text_trigramm(trigramme, startwortpaar=("die", "künstliche"), anzahl_woerter=20))

Vergleiche mehrere Durchläufe der Bigramm-Generierung (Schritt 4) mit mehreren Durchläufen der Trigramm-Generierung oben. Welche wirkt zusammenhängender? Notiere deine Beobachtung in ein bis zwei Sätzen (z. B. in einer eigenen Markdown-Zelle oder als Kommentar).

### Aufgabe 3: Eigenen Text einsetzen

Ersetze in Schritt 1 den Beispieltext durch einen eigenen, längeren deutschen Text (mindestens 80–100 Wörter, z. B. über ein Hobby, ein Buch oder ein Thema deiner Wahl – am besten mit einigen sich wiederholenden Wörtern oder Formulierungen). Führe danach alle Zellen von Schritt 2 bis Schritt 4 erneut aus. Wie verändert sich die Qualität des generierten Texts im Vergleich zum Beispieltext?

In [ ]:
# Hier deinen eigenen Text einsetzen und die Zellen von Schritt 2-4 erneut ausführen
# text = """Dein eigener Text hier..."""


## Zusammenfassung: Was ist gleich, was ist anders im Vergleich zu einem echten LLM?

**Gleich (das Grundprinzip):**
- Beide sagen Schritt für Schritt das wahrscheinlichste nächste Wort/Token voraus, basierend auf dem bisherigen Text.
- Beide lernen diese Wahrscheinlichkeiten ausschließlich aus Trainingsdaten (bei uns: der eine Beispieltext; bei einem LLM: Milliarden Textseiten).

**Anders:**
- Ein echtes LLM nutzt ein riesiges neuronales Netz (Transformer-Architektur, siehe die Theorie-Seite zu neuronalen Netzen), nicht reines Abzählen von Wortpaaren.
- Ein LLM berücksichtigt sehr viel mehr Kontext (oft tausende Wörter zurück, nicht nur 1-2 wie bei uns).
- Ein LLM wurde zusätzlich in einer Nachtrainingsphase darauf abgestimmt, hilfreich, sicher und in einem bestimmten Stil zu antworten – das haben wir hier nicht nachgebaut.

Trotzdem: Das Grundprinzip "Wahrscheinlichkeit für das nächste Wort" hast du jetzt nicht nur gelesen, sondern selbst programmiert und beobachtet.